# Algorithm 2: FilterCompTCT — RDP Filter Evaluation

This notebook evaluates **Algorithm 2 (FilterCompTCT)** from the paper.

## What Algorithm 2 does

FilterCompTCT runs adaptive TCT interleaved with non-TCT RDP queries against a **shared RDP budget** εRDP. The filter FILTTCT is checked **before each round i** using only the transcript through round i−1:

$$\text{FILTTCT}(y_1,\ldots,y_{i-1};\,\varepsilon'_1,\ldots,\varepsilon'_{i-1}) := \text{PASS if }\underbrace{K_{1+\sum_{s<i}y_s}\cdot\rho}_{\text{look-ahead reservation}} + \underbrace{\sum_{s<i}\varepsilon'_s}_{\text{non-TCT cost}} > \varepsilon_{\text{RDP}}$$

where:
- $y_s = 1$ iff round $s$ is a TCT round that produced a target hit
- $H = \sum_{s<i} y_s$ = cumulative hits before round $i$
- $K_{H+1}$ = Zeta-2 anytime schedule look-ahead (one full interval ahead)
- $\varepsilon'_s = 0$ for TCT rounds, $\varepsilon'_s > 0$ for non-TCT rounds

If PASS → output ⊥ (skip). Otherwise → execute $M_i(D)$ and update $y_i$.

**Key efficiency property:** TCT rounds are free between hits — the filter only advances the budget reservation at hit boundaries. The system answers many more rounds than a standard RDP filter when the empirical hit rate is small.

## Guarantee (Theorem 4)

Algorithm 2 satisfies $(\alpha, \varepsilon_{\text{RDP}}, \delta_\infty)$-RDP (approximate-RDP, Definition 2).

## Experiments

| Section | Question |
|---|---|
| 1 | Filter trace: how does the look-ahead reservation evolve? |
| 2 | Rounds answered vs εRDP budget |
| 3 | Rounds answered vs empirical hit rate |
| 4 | Mixed workload: impact of interleaved non-TCT rounds |

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 8.5,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.linewidth': 2.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ALG2_COLOR = '#2166ac'    # FilterCompTCT — blue
STD_COLOR  = '#d6604d'    # Standard RDP filter — red
GRAY       = '#888888'

In [ ]:
# ── Shared parameters ─────────────────────────────────────────────────────────
EPSILON      = 0.01      # ε for Laplace mechanism
DELTA_INF    = 1e-2      # δ∞ anytime failure probability (Zeta-2 schedule)
ALPHA_RDP    = 100.0     # Rényi order α
EPS_RDP      = 1.0       # shared RDP budget εRDP
HIT_RATE     = 0.02      # empirical fraction of TCT rounds producing a hit (2% << q)
N_ROUNDS     = 10_000    # max rounds per simulation run
SEED         = 42

rho = ALPHA_RDP * EPSILON**2 / 2.0   # per-call RDP cost (ε-DP → RDP conversion)
q   = 1.0 / (math.exp(EPSILON) + 1)  # NotPrior target probability

acc_ref = AdaptiveTCTAccountant(EPSILON, DELTA_INF, alpha_rdp=ALPHA_RDP)

# τ_max: max hit count Algorithm 2 admits before filter halts (pure TCT workload)
tau_max = 0
while acc_ref.compositions(tau_max + 2) * rho <= EPS_RDP:
    tau_max += 1

std_rounds_theory = EPS_RDP / rho   # standard RDP filter: εRDP / ρ rounds

print(f'Parameters: ε={EPSILON}, α={ALPHA_RDP}, ρ={rho:.6f}, εRDP={EPS_RDP}')
print(f'q = {q:.4f}   (NotPrior target, hit_rate={HIT_RATE} << q)')
print(f'δ∞ = {DELTA_INF:.0e}')
print()
print(f'Standard RDP filter: answers ~{std_rounds_theory:.0f} rounds')
print(f'FilterCompTCT τ_max: {tau_max} hits before filter halts')
print(f'Expected rounds (hit_rate={HIT_RATE}): ~{tau_max / HIT_RATE:.0f}')
print(f'Expected speedup: ~{tau_max / HIT_RATE / std_rounds_theory:.1f}×')

In [ ]:
# ── Algorithm 2 implementation ────────────────────────────────────────────────

def run_alg2(
    epsilon, delta_inf, alpha_rdp, eps_rdp,
    hit_rate, n_rounds, interleave_ratio=0.0, rho_non_tct=None, seed=SEED,
):
    """
    Simulate Algorithm 2 (FilterCompTCT).

    Filter condition (checked before round i, Alg 2 line 8):
      K_{1+H} * rho + acc_non_tct > eps_rdp  →  PASS (halt)

    TCT rounds:     ε'_i = 0  (free; only hit boundaries advance the budget)
    Non-TCT rounds: ε'_i = rho_non_tct  (charged immediately)

    Returns dict with simulation results and full trace.
    """
    rng = np.random.default_rng(seed)
    acc = AdaptiveTCTAccountant(epsilon, delta_inf, alpha_rdp=alpha_rdp)
    rho = alpha_rdp * epsilon**2 / 2.0
    if rho_non_tct is None:
        rho_non_tct = rho   # non-TCT rounds cost the same per-round as TCT

    H            = 0     # cumulative hits Σ y_s (s < i)
    acc_non_tct  = 0.0   # accumulated non-TCT cost Σ ε'_s (s < i)
    rounds_ans   = 0

    trace_H           = []
    trace_look_ahead  = []
    trace_non_tct     = []
    trace_total       = []

    for i in range(n_rounds):
        # Filter check: K_{1+H} * rho + acc_non_tct > eps_rdp  →  PASS
        look_ahead = acc.compositions(H + 1) * rho
        total      = look_ahead + acc_non_tct

        trace_H.append(H)
        trace_look_ahead.append(look_ahead)
        trace_non_tct.append(acc_non_tct)
        trace_total.append(total)

        if total > eps_rdp:
            break   # PASS: filter halts

        # Round executes
        rounds_ans += 1
        is_tct = (interleave_ratio == 0.0 or rng.random() >= interleave_ratio)

        if is_tct:
            if rng.random() < hit_rate:   # empirical hit
                H += 1
            # ε'_i = 0 (Alg 2 line 5)
        else:
            acc_non_tct += rho_non_tct    # ε'_i > 0 (Alg 2 line 7)

    return dict(
        rounds_answered=rounds_ans,
        hits=H,
        acc_non_tct=acc_non_tct,
        final_look_ahead=acc.compositions(H + 1) * rho,
        trace_H=trace_H,
        trace_look_ahead=trace_look_ahead,
        trace_non_tct=trace_non_tct,
        trace_total=trace_total,
    )


def run_standard_rdp(
    rho, eps_rdp, n_rounds, interleave_ratio=0.0, rho_non_tct=None, seed=SEED,
):
    """
    Standard RDP composition filter (no TCT).
    Charges rho per TCT round, rho_non_tct per non-TCT round.
    Halts when accumulated cost > eps_rdp.
    """
    rng = np.random.default_rng(seed)
    if rho_non_tct is None:
        rho_non_tct = rho
    accumulated = 0.0
    rounds_ans  = 0
    for _ in range(n_rounds):
        is_tct = (interleave_ratio == 0.0 or rng.random() >= interleave_ratio)
        cost   = rho if is_tct else rho_non_tct
        if accumulated + cost > eps_rdp:
            break
        accumulated += cost
        rounds_ans  += 1
    return rounds_ans


print('Functions defined: run_alg2(), run_standard_rdp()')

## Section 1: Filter Trace

How does the look-ahead reservation $K_{H+1}\cdot\rho$ evolve over rounds?

- Between hits: $H$ is constant → look-ahead is flat (rounds are free)
- At each hit: $H$ increments → look-ahead jumps by $\varphi_\alpha(H) = (K_{H+1} - K_H)\cdot\rho$ (Eq. 8)
- Filter halts when look-ahead crosses $\varepsilon_{\text{RDP}}$

In [ ]:
result = run_alg2(EPSILON, DELTA_INF, ALPHA_RDP, EPS_RDP,
                  HIT_RATE, N_ROUNDS, seed=SEED)

print(f'FilterCompTCT (pure TCT, hit_rate={HIT_RATE}):')
print(f'  Rounds answered: {result["rounds_answered"]:,}')
print(f'  Hits at halt:    {result["hits"]}')
print(f'  Final look-ahead K_{{H+1}}·ρ = {result["final_look_ahead"]:.4f}')
print(f'  εRDP budget:              {EPS_RDP:.4f}')

std_rounds = run_standard_rdp(rho, EPS_RDP, N_ROUNDS, seed=SEED)
print()
print(f'Standard RDP filter:       {std_rounds:,} rounds')
print(f'Speedup: {result["rounds_answered"] / std_rounds:.1f}×')

# ── Filter trace figure ───────────────────────────────────────────────────────
n_tr   = len(result['trace_total'])
rnds   = list(range(n_tr))

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)

# Top: look-ahead reservation over time
ax = axes[0]
ax.plot(rnds, result['trace_look_ahead'], color=ALG2_COLOR, lw=1.5,
        label=r'Look-ahead $K_{H+1}\cdot\rho$')
ax.axhline(EPS_RDP, color='black', lw=0.9, linestyle='--',
           label=rf'$\varepsilon_{{\mathrm{{RDP}}}} = {EPS_RDP}$')
ax.set_ylabel('RDP cost')
ax.set_title('Look-ahead reservation (jumps at each hit, flat between hits)')
ax.legend(loc='upper left', fontsize=8)
ax.set_ylim(0, EPS_RDP * 1.15)

# Bottom: cumulative hit count
ax = axes[1]
ax.plot(rnds, result['trace_H'], color=ALG2_COLOR, lw=1.5,
        label='Cumulative hits $H$')
ax.set_xlabel('Query round')
ax.set_ylabel('Hit count $H$')
ax.set_title(f'Cumulative hits (halt at H={result["hits"]})')
ax.legend(loc='upper left', fontsize=8)

fig.suptitle(
    rf'Algorithm 2 Filter Trace — $\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, '
    rf'$\varepsilon_{{\mathrm{{RDP}}}}$={EPS_RDP}, $\delta_\infty$={DELTA_INF:.0e}, hit rate={HIT_RATE}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/alg2_filter_trace.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 2: Rounds Answered vs εRDP Budget

For a standard RDP filter, rounds answered = εRDP / ρ (linear in budget).

For FilterCompTCT, the binding constraint is τ_max (max hits before filter halts), and rounds ≈ τ_max / hit_rate.
τ_max grows roughly as K_{τmax+1} · ρ ≤ εRDP, so τ_max ≈ εRDP / ρ · q / (1+α_τmax) — still linear in εRDP,
but the *rounds* scale as τ_max / hit_rate >> τ_max when hit_rate << q.

In [ ]:
budgets    = np.linspace(0.05, 3.0, 40)
alg2_rounds, std_rounds_list = [], []

for eps_b in budgets:
    r = run_alg2(EPSILON, DELTA_INF, ALPHA_RDP, eps_b, HIT_RATE, N_ROUNDS, seed=SEED)
    alg2_rounds.append(r['rounds_answered'])
    std_rounds_list.append(run_standard_rdp(rho, eps_b, N_ROUNDS, seed=SEED))

# Theoretical standard line
std_theory = budgets / rho

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))

ax = axes[0]
ax.plot(budgets, alg2_rounds,     color=ALG2_COLOR, lw=2,
        label='FilterCompTCT (Alg 2)')
ax.plot(budgets, std_rounds_list,  color=STD_COLOR,  lw=2, linestyle='--',
        label='Standard RDP filter')
ax.plot(budgets, std_theory,       color=STD_COLOR,  lw=1, linestyle=':',
        label=r'Theory: $\varepsilon_{\mathrm{RDP}} / \rho$')
ax.set_xlabel(r'RDP budget $\varepsilon_{\mathrm{RDP}}$')
ax.set_ylabel('Rounds answered before filter halts')
ax.set_title('Rounds answered vs budget')
ax.legend(fontsize=8)

ax = axes[1]
speedup = np.array(alg2_rounds) / np.maximum(std_rounds_list, 1)
ax.plot(budgets, speedup, color=ALG2_COLOR, lw=2)
ax.axhline(1, color=STD_COLOR, lw=0.8, linestyle='--')
ax.set_xlabel(r'RDP budget $\varepsilon_{\mathrm{RDP}}$')
ax.set_ylabel('FilterCompTCT / Standard')
ax.set_title(f'Speedup (hit rate = {HIT_RATE})')

fig.suptitle(
    rf'Algorithm 2 vs Standard RDP — $\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, '
    rf'$\\delta_\\infty$={DELTA_INF:.0e}, hit rate={HIT_RATE}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/alg2_rounds_vs_budget.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 3: Rounds Answered vs Empirical Hit Rate

The standard filter answers εRDP/ρ rounds regardless of hit rate.

FilterCompTCT answers ≈ τ_max / hit_rate rounds. As hit_rate → 0 (rare threshold crossings),
FilterCompTCT's advantage grows without bound — the TCT-favourable regime.

In [ ]:
hit_rates  = np.linspace(0.005, 0.40, 50)
alg2_by_hr = []

for hr in hit_rates:
    r = run_alg2(EPSILON, DELTA_INF, ALPHA_RDP, EPS_RDP, hr, N_ROUNDS, seed=SEED)
    alg2_by_hr.append(r['rounds_answered'])

std_flat = run_standard_rdp(rho, EPS_RDP, N_ROUNDS, seed=SEED)

# Mark the q threshold
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))

ax = axes[0]
ax.plot(hit_rates, alg2_by_hr, color=ALG2_COLOR, lw=2,
        label='FilterCompTCT (Alg 2)')
ax.axhline(std_flat, color=STD_COLOR, lw=2, linestyle='--',
           label=f'Standard RDP ({std_flat:,} rounds)')
ax.axvline(q, color=GRAY, lw=1, linestyle=':',
           label=rf'$q = 1/(e^{{\varepsilon}}+1) \approx {q:.3f}$')
ax.set_xlabel('Empirical hit rate')
ax.set_ylabel('Rounds answered')
ax.set_title('Rounds answered vs hit rate')
ax.legend(fontsize=8)

ax = axes[1]
speedup_hr = np.array(alg2_by_hr) / max(std_flat, 1)
ax.plot(hit_rates, speedup_hr, color=ALG2_COLOR, lw=2)
ax.axhline(1, color=STD_COLOR, lw=0.8, linestyle='--')
ax.axvline(q, color=GRAY, lw=1, linestyle=':',
           label=rf'$q \approx {q:.3f}$')
ax.set_xlabel('Empirical hit rate')
ax.set_ylabel('FilterCompTCT / Standard')
ax.set_title('Speedup factor')
ax.legend(fontsize=8)

fig.suptitle(
    rf'Algorithm 2 Advantage vs Hit Rate — $\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, '
    rf'$\varepsilon_{{\mathrm{{RDP}}}}$={EPS_RDP}, $\delta_\infty$={DELTA_INF:.0e}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/alg2_rounds_vs_hitrate.png', bbox_inches='tight', dpi=200)
plt.show()

# Summary table
print(f'Standard RDP: {std_flat:,} rounds (flat across hit rates)')
print(f'\nFilterCompTCT rounds answered:')
for hr in [0.01, 0.02, 0.05, 0.10, 0.20, float(q)]:
    idx = np.argmin(np.abs(hit_rates - hr))
    r   = alg2_by_hr[idx]
    print(f'  hit_rate={hr:.3f}: {r:5,} rounds  ({r/std_flat:.1f}× speedup)')

## Section 4: Mixed Workload — Interleaved Non-TCT Rounds

When non-TCT rounds are interleaved, they charge $\varepsilon'_i = \rho_{\text{non-TCT}}$ directly against the budget.
This reduces the remaining budget available for TCT look-ahead, shrinking the rounds-answered advantage.

The question: at what interleave ratio does the TCT advantage disappear?

In [ ]:
interleave_ratios = np.linspace(0.0, 0.9, 30)
alg2_by_intlv, std_by_intlv = [], []

for ir in interleave_ratios:
    r = run_alg2(EPSILON, DELTA_INF, ALPHA_RDP, EPS_RDP,
                 HIT_RATE, N_ROUNDS, interleave_ratio=ir, seed=SEED)
    alg2_by_intlv.append(r['rounds_answered'])
    std_by_intlv.append(
        run_standard_rdp(rho, EPS_RDP, N_ROUNDS, interleave_ratio=ir, seed=SEED)
    )

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))

ax = axes[0]
ax.plot(interleave_ratios * 100, alg2_by_intlv, color=ALG2_COLOR, lw=2,
        label='FilterCompTCT (Alg 2)')
ax.plot(interleave_ratios * 100, std_by_intlv,  color=STD_COLOR,  lw=2, linestyle='--',
        label='Standard RDP filter')
ax.set_xlabel('Non-TCT round fraction (%)')
ax.set_ylabel('Rounds answered')
ax.set_title('Impact of interleaved non-TCT rounds')
ax.legend(fontsize=8)

ax = axes[1]
speedup_intlv = np.array(alg2_by_intlv) / np.maximum(std_by_intlv, 1)
ax.plot(interleave_ratios * 100, speedup_intlv, color=ALG2_COLOR, lw=2)
ax.axhline(1, color=STD_COLOR, lw=0.8, linestyle='--', label='Breakeven')
ax.set_xlabel('Non-TCT round fraction (%)')
ax.set_ylabel('FilterCompTCT / Standard')
ax.set_title('Speedup vs interleave ratio')
ax.legend(fontsize=8)

fig.suptitle(
    rf'Algorithm 2 Mixed Workload — $\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, '
    rf'$\\delta_\\infty$={DELTA_INF:.0e}, hit rate={HIT_RATE}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/alg2_interleave.png', bbox_inches='tight', dpi=200)
plt.show()

print(f'Speedup at interleave ratios:')
for ir, sp in zip([0.0, 0.25, 0.50, 0.75, 0.90],
                  [speedup_intlv[np.argmin(np.abs(interleave_ratios - v))]
                   for v in [0.0, 0.25, 0.50, 0.75, 0.90]]):
    print(f'  {int(ir*100):>3}% non-TCT:  {sp:.2f}×')

## Section 5: Correctness — Filter Never Exceeds εRDP Budget

The filter halts *before* the RDP budget is exceeded. Verify that
$K_{H+1}\cdot\rho + \Sigma\varepsilon'_s \leq \varepsilon_{\text{RDP}}$ holds at the final answered round,
across many random seeds.

In [ ]:
N_SEEDS = 500
violations = 0
final_costs = []

for s in range(N_SEEDS):
    r = run_alg2(EPSILON, DELTA_INF, ALPHA_RDP, EPS_RDP, HIT_RATE, N_ROUNDS, seed=s)
    # Cost at the last answered round (before filter halt)
    if len(r['trace_total']) > 0:
        last_cost = r['trace_total'][-1]   # cost that triggered the halt
        prev_cost = r['trace_total'][-2] if len(r['trace_total']) > 1 else 0.0
        final_costs.append(prev_cost)      # cost at last *answered* round
        if prev_cost > EPS_RDP:
            violations += 1

print(f'Budget correctness check over {N_SEEDS} seeds:')
print(f'  Violations (cost > εRDP at last answered round): {violations}')
print(f'  Max final cost: {max(final_costs):.6f}  (εRDP = {EPS_RDP})')
print(f'  Mean final cost: {np.mean(final_costs):.4f}  ({100*np.mean(final_costs)/EPS_RDP:.1f}% utilization)')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(final_costs, bins=40, color=ALG2_COLOR, alpha=0.75, edgecolor='white')
ax.axvline(EPS_RDP, color='black', lw=1.5, linestyle='--',
           label=rf'$\varepsilon_{{\mathrm{{RDP}}}} = {EPS_RDP}$')
ax.set_xlabel(r"$K_{H+1}\cdot\rho + \Sigma\varepsilon'_s$ at last answered round")
ax.set_ylabel('Count')
ax.set_title(f'Budget utilization at filter halt ({N_SEEDS} runs)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/alg2_budget_utilization.png', bbox_inches='tight', dpi=200)
plt.show()